# Stage 5a (Tier 1b) — XLM Singlish Patch

Re-scores Singlish-containing chunks with `cardiffnlp/twitter-xlm-roberta-base-sentiment`
and patches them back into `chunk_sentiment.parquet`.

**Why:** `twitter-roberta-base-sentiment-latest` scores 40% accuracy on controlled Singlish
sentences; XLM scores 80%. XLM handles Singlish because its multilingual pretraining
covers Malay vocabulary that overlaps with Singlish particles and loanwords.

**What changes:** only chunks with ≥1 Singlish marker (word-boundary regex) are re-scored
— ~34k chunks (~4.7% of corpus). All other chunks keep their existing RoBERTa scores.

**Input datasets needed:**
- Output from `kaggle_sentiment_v1` — `chunk_sentiment.parquet` (unpatched)
- `ns-sentiment-chunks-v3` — `submissions_chunks.parquet`, `comments_chunks.parquet` (for chunk text)

**Output:** `chunk_sentiment.parquet` — same schema, Singlish rows replaced with XLM scores

**Setup:** GPU T4 x2 · Save & Run All

In [ ]:
# Cell 1 — Discover input paths
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# Cell 2 — Config  <- UPDATE PATHS AFTER RUNNING CELL 1
CHUNK_SENTIMENT    = '/kaggle/input/<sentiment-v1-output>/chunk_sentiment.parquet'
SUBMISSIONS_CHUNKS = '/kaggle/input/<your-dataset>/submissions_chunks.parquet'
COMMENTS_CHUNKS    = '/kaggle/input/<your-dataset>/comments_chunks.parquet'
OUT_DIR            = '/kaggle/working'

MODEL_NAME   = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
BATCH_SIZE   = 128
CHECKPOINT_N = 10_000

In [ ]:
# Cell 3 — Imports
!pip install transformers -q

import re
import time
import numpy as np
import pandas as pd
import torch
from transformers import pipeline

print('Imports OK')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 4 — Singlish detector (word-boundary regex)
_SINGLISH_VOCAB = {
    # particles
    'lah','leh','lor','liao','sia','hor','mah','bah','wah','nia',
    # sentiment terms
    'shiok','song','swee','steady','lobang','slack','lepak','sian','jialat',
    'wayang','chao keng','saikang','siong','tekan','kns','suay','teruk',
    'terok','bo chap','bochap','arrow','gg','bo liao','tok kok','cmi','kena',
    # extended
    'walao','walau','siao','aiyah','aiyoh','alamak','cheem','liddat',
    'shiok leh','tok gong','powderful','gao gao','die die',
}

# Longest-match first to avoid partial hits on multi-word phrases
_PATTERN = re.compile(
    r'\b(' + '|'.join(re.escape(w) for w in sorted(_SINGLISH_VOCAB, key=len, reverse=True)) + r')\b'
)

def has_singlish(text: str) -> bool:
    return bool(_PATTERN.search(str(text).lower()))

# Sanity check
assert has_singlish('sian lah kena guard duty') == True
assert has_singlish('this is a normal sentence') == False
print('Singlish detector OK')

In [ ]:
# Cell 5 — Load data
print('Loading chunk_sentiment.parquet ...')
sent = pd.read_parquet(CHUNK_SENTIMENT)
print(f'  {len(sent):,} rows  |  cols: {sent.columns.tolist()}')

print('Loading chunk text ...')
sub    = pd.read_parquet(SUBMISSIONS_CHUNKS, columns=['chunk_id','text'])
com    = pd.read_parquet(COMMENTS_CHUNKS,    columns=['chunk_id','text'])
chunks = pd.concat([sub, com], ignore_index=True).drop_duplicates(subset='chunk_id')
del sub, com
print(f'  {len(chunks):,} unique chunks loaded')

# Merge text into sentiment df
df = sent.merge(chunks[['chunk_id','text']], on='chunk_id', how='left')
del chunks
print(f'  Merge done — {df["text"].isna().sum():,} chunks missing text (expected ~0)')

In [ ]:
# Cell 6 — Identify Singlish chunks
print('Detecting Singlish chunks ...')
df['has_sg'] = df['text'].fillna('').apply(has_singlish)
sg_mask = df['has_sg']
n_sg    = sg_mask.sum()
print(f'  Singlish chunks: {n_sg:,} / {len(df):,}  ({n_sg/len(df)*100:.1f}%)')
print(f'  English chunks (unchanged): {(~sg_mask).sum():,}')

In [ ]:
# Cell 7 — Load XLM model
device = 0 if torch.cuda.is_available() else -1
print(f'Loading {MODEL_NAME}  (device={"GPU" if device==0 else "CPU"}) ...')

xlm = pipeline(
    'sentiment-analysis',
    model=MODEL_NAME,
    top_k=None,
    device=device,
    truncation=True,
    max_length=512,
)
print('Model loaded')

# Verify label set
test_out = xlm(['test'])
labels   = sorted(s['label'] for s in test_out[0])
print(f'Label set: {labels}')
assert set(labels) == {'negative','neutral','positive'}, f'Unexpected labels: {labels}'

In [ ]:
# Cell 8 — Batch inference on Singlish chunks
sg_df     = df[sg_mask].copy().reset_index(drop=True)
texts     = sg_df['text'].fillna('').str.strip().tolist()
chunk_ids = sg_df['chunk_id'].tolist()
n         = len(texts)

new_records      = []
checkpoint_files = []
t0               = time.time()

print(f'Scoring {n:,} Singlish chunks with XLM ...')

for start in range(0, n, BATCH_SIZE):
    batch_texts = texts[start : start + BATCH_SIZE]
    batch_ids   = chunk_ids[start : start + BATCH_SIZE]
    raw         = xlm(batch_texts, batch_size=BATCH_SIZE)

    for cid, scores in zip(batch_ids, raw):
        score_map = {s['label']: s['score'] for s in scores}
        new_records.append({
            'chunk_id': cid,
            'sent_neg': score_map.get('negative', np.nan),
            'sent_neu': score_map.get('neutral',  np.nan),
            'sent_pos': score_map.get('positive', np.nan),
        })

    done    = start + len(batch_texts)
    elapsed = time.time() - t0
    rate    = done / elapsed if elapsed > 0 else 0
    eta     = (n - done) / rate if rate > 0 else 0

    # Checkpoint
    if done % CHECKPOINT_N < BATCH_SIZE or done >= n:
        cp_path = f'{OUT_DIR}/xlm_checkpoint_{done}.parquet'
        pd.DataFrame(new_records).to_parquet(cp_path, index=False)
        checkpoint_files.append(cp_path)
        print(f'  {done:>6,} / {n:,}  |  {rate:.0f} chunks/s  |  ETA {eta/60:.1f} min  |  checkpoint saved')

print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 9 — Merge checkpoints, patch original, save
cp_dfs  = [pd.read_parquet(f) for f in checkpoint_files]
new_df  = pd.concat(cp_dfs, ignore_index=True).drop_duplicates(subset='chunk_id')
print(f'XLM-scored chunks: {len(new_df):,}  (expected ~{n_sg:,})')

# Patch: update only the rows whose chunk_id was re-scored
patched = sent.set_index('chunk_id')
patched.update(new_df.set_index('chunk_id'))
patched = patched.reset_index()

out_path = f'{OUT_DIR}/chunk_sentiment.parquet'
patched.to_parquet(out_path, index=False)
print(f'Saved patched parquet → {out_path}')
print(f'Total rows: {len(patched):,}  (should match original {len(sent):,})')

In [ ]:
# Cell 10 — Summary
label_map = {'sent_neg':'negative','sent_neu':'neutral','sent_pos':'positive'}
patched['majority_label'] = patched[['sent_neg','sent_neu','sent_pos']].idxmax(axis=1).map(label_map)

print('── Full corpus label distribution after patch ──')
vc = patched['majority_label'].value_counts()
for lbl, cnt in vc.items():
    print(f'  {lbl:<10} {cnt:>8,}  ({cnt/len(patched)*100:.1f}%)')

print('\n── XLM scores on patched Singlish chunks only ──')
sg_ids    = new_df['chunk_id'].values
sg_subset = patched[patched['chunk_id'].isin(sg_ids)]
sg_labels = sg_subset[['sent_neg','sent_neu','sent_pos']].idxmax(axis=1).map(label_map)
sg_vc     = sg_labels.value_counts()
for lbl, cnt in sg_vc.items():
    print(f'  {lbl:<10} {cnt:>8,}  ({cnt/len(sg_subset)*100:.1f}%)')

print('\nDone. Download chunk_sentiment.parquet from /kaggle/working.')